# Liu2024 — Slim subject-adaptive calibration transfer

This notebook is a stricter rerun of the adaptive-calibration experiment.

The goal is **not** to train a new S-JEPA and not to use cached embeddings. The S-JEPA branch uses the official `braindecode/signal-jepa_without-chans` pretrained weights, which were trained on healthy-subject EEG. This tests whether that representation transfers to Liu2024 acute-stroke motor imagery.

Main corrections compared with the previous notebook:

1. **Fold-safe target Euclidean Alignment (EA).**  
   Target-subject EA is fitted only on the target calibration trials for `within_only` and `transfer_plus_cal`. Test trials are never used to estimate the target alignment transform.

2. **Strict zero-shot transfer.**  
   `transfer_only` does not use target calibration trials. Its target test data are not target-EA aligned.

3. **Adaptive sweep included.**  
   `transfer_plus_cal` sweeps:
   - calibration-trial upweighting
   - source-pool subsampling

4. **Slim configuration.**  
   Removed training, augmentation, downstream fine-tuning, checkpointing, and unused S-JEPA training options. This notebook only does preprocessing, official pretrained S-JEPA embedding extraction, Riemannian features, fusion, and calibration-transfer evaluation.


In [1]:
import os, re, json, hashlib, random, platform, sys, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.linalg import eigh

from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print(f"[setup] matplotlib unavailable: {exc}")

try:
    import mne
    mne.set_log_level("WARNING")
    HAVE_MNE = True
except Exception as exc:
    HAVE_MNE = False
    print(f"[setup] mne unavailable: {exc}")

try:
    import torch
    HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False
    print(f"[setup] torch unavailable: {exc}")

try:
    from braindecode.models import SignalJEPA_PreLocal
    HAVE_BRAINDECODE = True
except Exception as exc:
    HAVE_BRAINDECODE = False
    print(f"[setup] braindecode unavailable; S-JEPA branch cannot run: {exc}")

try:
    from pyriemann.estimation import Covariances
    HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False
    print(f"[setup] pyriemann unavailable; using numpy covariance fallback: {exc}")

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("deps:", {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "mne": HAVE_MNE,
    "torch": HAVE_TORCH,
    "braindecode": HAVE_BRAINDECODE,
    "pyriemann": HAVE_PYRIEMANN,
})


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


deps: {'python': '3.11.15', 'platform': 'Linux-6.14.0-37-generic-x86_64-with-glibc2.41', 'mne': True, 'torch': True, 'braindecode': True, 'pyriemann': True}


## 1. Minimal configuration


In [2]:
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for c in candidates:
        if (c / "liu2024_data").exists() or (c / "artifacts").exists():
            return c
    # fallback used by older notebooks launched from notebooks/*/*
    return start.parent.parent

PROJECT_ROOT = find_project_root()

RUN = {
    "experiment_name": "subject_adaptive_calibration_transfer_slim",
    "artifact_root": str(PROJECT_ROOT / "artifacts" / "liu2024-subject-adaptive-calibration-slim"),
    "source_extract_dir": str(PROJECT_ROOT / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "seed": 2026,
}

PREPROCESS = {
    # Liu source MAT values are usually microvolts. MNE filtering uses volts internally.
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # Same cleaned setup as your previous adaptive run.
    "resample_sfreq": 128.0,
    "reference": "average",
    "filter_low": 0.5,
    "filter_high": 40.0,

    # Keep the same S-JEPA-compatible window you were already using.
    "mi_window_start_s": 1.5,
    "target_window_samples": 537,
}

SJEPA = {
    # Official healthy-subject pretrained S-JEPA weights from the authors/Braindecode.
    # No local embedding path is used in this notebook.
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "embedding_module": "final_layer",
    "embedding_pool": "mean",
    "embedding_batch": 64,
}

ADAPT = {
    # Start with the same decodable diagnostic subgroup. Change to "all" when the corrected
    # logic looks sane and you are ready for the full 50-subject run.
    "target_subjects": [7, 22, 23, 28, 40, 44],

    "branches": ["riemann", "sjepa", "fusion"],
    "calibration_grid": [0, 4, 8, 12, 16, 24],
    "n_repeats": 10,

    # The actual adaptive sweep.
    # source_cap = number of non-target source trials used by transfer strategies.
    # "all" means all non-target trials.
    "source_cap_grid": [24, 48, 96, 200, "all"],

    # Used only for transfer_plus_cal. Calibration trials are upweighted against source trials.
    "calibration_upweight_grid": [1, 5, 10, 25, 50, 100],

    # Classifier/feature options.
    "cov_estimator": "oas",
    "standardize": True,
    "logreg_C": 1.0,
    "logreg_max_iter": 2000,
}

def make_run_id():
    stamp = datetime.now().strftime("%Y%m%d_%H%M")
    h = hashlib.md5(json.dumps({"RUN": RUN, "PREPROCESS": PREPROCESS, "SJEPA": SJEPA, "ADAPT": ADAPT}, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{stamp}_{h}"

RUN_ID = make_run_id()
ARTIFACT_DIR = Path(RUN["artifact_root"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump({"RUN": RUN, "PREPROCESS": PREPROCESS, "SJEPA": SJEPA, "ADAPT": ADAPT}, f, indent=2, default=str)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("SOURCE_EXTRACT_DIR:", RUN["source_extract_dir"])


PROJECT_ROOT: /home/vegorov/Repos/eeg_jepa_research
ARTIFACT_DIR: /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-subject-adaptive-calibration-slim/20260614_1409_10bb0fb8
SOURCE_EXTRACT_DIR: /home/vegorov/Repos/eeg_jepa_research/liu2024_data/liu2024_figshare/sourcedata


In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    if HAVE_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass

def resolve_device():
    if not HAVE_TORCH:
        return None
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

seed_everything(int(RUN["seed"]))
DEVICE = resolve_device()
print("DEVICE:", DEVICE)


DEVICE: cpu


## 2. Liu2024 channel conventions


In [4]:
LIU_SOURCE_SFREQ = 500.0
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Source MAT: 0..29 EEG-like channels, channel 17 is CPz source reference,
# 30..31 EOG, 32 marker. We drop CPz, EOG, and marker.
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
EEG_CHANNEL_INDICES = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]
N_CHANS = len(EEG_CHANNEL_NAMES)
N_CLASSES = 2

print(f"Using {N_CHANS} EEG channels:")
print(EEG_CHANNEL_NAMES)


Using 29 EEG channels:
['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']


## 3. Robust source-MAT loading


In [5]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({"name": name, "type": "ndarray", "shape": str(value.shape), "dtype": str(value.dtype)})
        else:
            rows.append({"name": name, "type": type(value).__name__, "shape": "", "dtype": ""})
    return pd.DataFrame(rows).head(max_rows)

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")

    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}. Preview saved to {preview_path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name

def validate_liu_subject(rawdata, labels, subject_id, path=None):
    if rawdata.shape[0] != labels.size:
        raise ValueError(f"Subject {subject_id}: rawdata/labels mismatch: {rawdata.shape} vs {labels.shape}, path={path}")
    if rawdata.shape[0] != LIU_EXPECTED_TRIALS_PER_SUBJECT:
        print(f"WARNING subject {subject_id}: expected 40 trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] != LIU_EXPECTED_SOURCE_CHANNELS:
        print(f"WARNING subject {subject_id}: expected 33 source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL:
        print(f"WARNING subject {subject_id}: expected 4000 samples/trial, got {rawdata.shape[2]}")
    y = labels_to_zero_based(labels)
    counts = np.bincount(y, minlength=N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: one class is missing after label conversion: {counts.tolist()}")


## 4. Minimal preprocessing


In [6]:
def make_mne_info(sfreq):
    if not HAVE_MNE:
        raise RuntimeError("mne is required for this preprocessing pipeline.")
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def source_to_volts(X):
    unit = PREPROCESS["source_unit"].lower()
    X = np.asarray(X, dtype=np.float64)
    if unit in {"microvolts", "uv", "µv"}:
        return X * 1e-6
    if unit in {"volts", "v"}:
        return X
    raise ValueError(f"Unsupported source_unit={PREPROCESS['source_unit']}")

def volts_to_model_unit(X):
    unit = PREPROCESS["final_model_unit"].lower()
    X = np.asarray(X, dtype=np.float64)
    if unit in {"microvolts", "uv", "µv"}:
        return X * 1e6
    if unit in {"volts", "v"}:
        return X
    raise ValueError(f"Unsupported final_model_unit={PREPROCESS['final_model_unit']}")

def preprocess_subject(rawdata, labels, subject_id):
    # 1) select 29 EEG channels only
    X = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)

    # 2) MNE wants continuous channels x samples in volts
    X_volts = source_to_volts(X)
    continuous = X_volts.transpose(1, 0, 2).reshape(N_CHANS, -1)

    raw = mne.io.RawArray(continuous, make_mne_info(LIU_SOURCE_SFREQ), verbose=False)

    # 3) average reference -> resample -> 0.5-40 Hz FIR
    if PREPROCESS["reference"] == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
    elif PREPROCESS["reference"] not in {None, "none"}:
        raise ValueError(f"Unsupported reference={PREPROCESS['reference']}")

    raw.resample(float(PREPROCESS["resample_sfreq"]), verbose=False)
    raw.filter(
        l_freq=float(PREPROCESS["filter_low"]),
        h_freq=float(PREPROCESS["filter_high"]),
        method="fir",
        phase="zero",
        fir_design="firwin",
        verbose=False,
    )

    # 4) back to trials x channels x samples, in model unit
    data = volts_to_model_unit(raw.get_data())
    n_trials = rawdata.shape[0]
    effective_sfreq = float(raw.info["sfreq"])
    samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / LIU_SOURCE_SFREQ))

    # MNE resampling can introduce one-sample rounding differences; trim to full trials.
    if data.shape[1] < n_trials * samples_per_trial:
        samples_per_trial = data.shape[1] // n_trials
    data = data[:, :n_trials * samples_per_trial]
    X_rs = data.reshape(N_CHANS, n_trials, samples_per_trial).transpose(1, 0, 2)

    start = int(round(float(PREPROCESS["mi_window_start_s"]) * effective_sfreq))
    stop = start + int(PREPROCESS["target_window_samples"])
    if stop > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start}:{stop}] exceeds trial length {X_rs.shape[-1]}."
        )

    X_win = X_rs[:, :, start:stop].astype(np.float32)
    y = labels_to_zero_based(labels).astype(np.int64)

    return X_win, y, {
        "subject_id": int(subject_id),
        "n_trials": int(len(y)),
        "class_counts": np.bincount(y, minlength=N_CLASSES).tolist(),
        "samples_per_trial_after_resample": int(samples_per_trial),
        "crop_start_sample": int(start),
        "crop_stop_sample": int(stop),
        "window_samples": int(X_win.shape[-1]),
        "effective_sfreq": float(effective_sfreq),
    }


## 5. Load and preprocess all subjects


In [7]:
SOURCE_EXTRACT_DIR = Path(RUN["source_extract_dir"])
MAT_FILES = find_source_mat_files(SOURCE_EXTRACT_DIR)

if not MAT_FILES:
    raise FileNotFoundError(f"No .mat files found under {SOURCE_EXTRACT_DIR}")

print(f"Found {len(MAT_FILES)} MAT files under {SOURCE_EXTRACT_DIR}")

# Save preview for audit/debugging.
preview = mat_structure_preview(MAT_FILES[0])
preview.to_csv(ARTIFACT_DIR / "mat_structure_preview_first_subject.csv", index=False)

subjects_meta = []
Xs, ys, subject_ids = [], [], []

for p in MAT_FILES:
    sid = subject_id_from_path(p)
    rawdata, labels, raw_field, label_field = load_subject_mat(p)
    validate_liu_subject(rawdata, labels, sid, path=p)

    X_win, y, prep_stats = preprocess_subject(rawdata, labels, sid)

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))

    subjects_meta.append({
        "subject_id": sid,
        "path": str(p),
        "raw_field": raw_field,
        "label_field": label_field,
        "rawdata_shape": tuple(rawdata.shape),
        "labels_shape": tuple(labels.shape),
        **prep_stats,
    })

subjects_df = pd.DataFrame(subjects_meta).sort_values("subject_id").reset_index(drop=True)
subjects_df.to_csv(ARTIFACT_DIR / "subject_inventory.csv", index=False)

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0).astype(int)
SUBJ = np.asarray(subject_ids).astype(str)

subjects_all = sorted(set(SUBJ), key=lambda s: int(s))
print("subjects:", subjects_all)
print("X_ALL:", X_ALL.shape, "| labels:", np.bincount(Y_ALL).tolist())

display(subjects_df[[
    "subject_id", "n_trials", "class_counts", "window_samples",
    "crop_start_sample", "crop_stop_sample", "effective_sfreq"
]].head())


Found 50 MAT files under /home/vegorov/Repos/eeg_jepa_research/liu2024_data/liu2024_figshare/sourcedata
subjects: [np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'), np.str_('9'), np.str_('10'), np.str_('11'), np.str_('12'), np.str_('13'), np.str_('14'), np.str_('15'), np.str_('16'), np.str_('17'), np.str_('18'), np.str_('19'), np.str_('20'), np.str_('21'), np.str_('22'), np.str_('23'), np.str_('24'), np.str_('25'), np.str_('26'), np.str_('27'), np.str_('28'), np.str_('29'), np.str_('30'), np.str_('31'), np.str_('32'), np.str_('33'), np.str_('34'), np.str_('35'), np.str_('36'), np.str_('37'), np.str_('38'), np.str_('39'), np.str_('40'), np.str_('41'), np.str_('42'), np.str_('43'), np.str_('44'), np.str_('45'), np.str_('46'), np.str_('47'), np.str_('48'), np.str_('49'), np.str_('50')]
X_ALL: (2000, 29, 537) | labels: [1000, 1000]


,subject_id,n_trials,class_counts,window_samples,crop_start_sample,crop_stop_sample,effective_sfreq
0,1,40,"[20, 20]",537,192,729,128.0
1,2,40,"[20, 20]",537,192,729,128.0
2,3,40,"[20, 20]",537,192,729,128.0
3,4,40,"[20, 20]",537,192,729,128.0
4,5,40,"[20, 20]",537,192,729,128.0


## 6. Features: fold-safe EA, Riemannian tangent features, official S-JEPA embeddings


In [8]:
def _regularize_spd(C, eps=1e-6):
    C = np.asarray(C, dtype=np.float64)
    tr = float(np.trace(C))
    scale = tr / C.shape[0] if tr > 0 else 1.0
    return C + eps * scale * np.eye(C.shape[0])

def _inv_sqrt_spd(R, eps=1e-8):
    R = _regularize_spd(R, eps=eps)
    w, V = eigh(R)
    w = np.clip(w, eps, None)
    return (V * (1.0 / np.sqrt(w))) @ V.T

def _logm_spd(C, eps=1e-12):
    C = _regularize_spd(C, eps=eps)
    w, V = eigh(C)
    w = np.clip(w, eps, None)
    return (V * np.log(w)) @ V.T

def trial_covariances(X):
    X = np.asarray(X, dtype=np.float64)
    if HAVE_PYRIEMANN:
        return Covariances(estimator=ADAPT["cov_estimator"]).transform(X)
    n, c, t = X.shape
    out = np.empty((n, c, c), dtype=np.float64)
    for i in range(n):
        Xi = X[i] - X[i].mean(axis=1, keepdims=True)
        C = Xi @ Xi.T / max(t - 1, 1)
        out[i] = _regularize_spd(C)
    return out

def fit_ea_transform(X_fit):
    X_fit = np.asarray(X_fit, dtype=np.float64)
    if len(X_fit) == 0:
        return np.eye(N_CHANS)
    covs = np.einsum("nct,ndt->ncd", X_fit, X_fit) / X_fit.shape[2]
    R = covs.mean(axis=0)
    return _inv_sqrt_spd(R)

def apply_ea_transform(X, P):
    return np.einsum("cd,ndt->nct", P, np.asarray(X, dtype=np.float64))

def tangent_at_identity(covs):
    c = covs.shape[1]
    iu = np.triu_indices(c)
    scale = np.sqrt(2.0) * np.ones((c, c), dtype=np.float64)
    np.fill_diagonal(scale, 1.0)
    out = np.empty((len(covs), len(iu[0])), dtype=np.float64)
    for i, C in enumerate(covs):
        L = _logm_spd(C) * scale
        out[i] = L[iu]
    return out

def build_official_sjepa_model():
    if not (HAVE_TORCH and HAVE_BRAINDECODE):
        raise RuntimeError("torch and braindecode are required for the official S-JEPA branch.")

    model = SignalJEPA_PreLocal.from_pretrained(
        SJEPA["pretrained_repo_id"],
        n_chans=N_CHANS,
        n_times=int(PREPROCESS["target_window_samples"]),
        n_outputs=N_CLASSES,
        chs_info=make_mne_info(float(PREPROCESS["resample_sfreq"]))["chs"],
        strict=False,
    )
    model.eval().to(DEVICE)
    return model

def get_submodule_by_suffix(model, name):
    for n, m in model.named_modules():
        if n == name or n.endswith("." + name):
            return m
    return None

SJEPA_MODEL = None
SJEPA_AVAILABLE = False

if "sjepa" in ADAPT["branches"] or "fusion" in ADAPT["branches"]:
    try:
        SJEPA_MODEL = build_official_sjepa_model()
        SJEPA_AVAILABLE = True
        print(f"S-JEPA official pretrained model loaded: {SJEPA['pretrained_repo_id']}")
    except Exception as exc:
        print(f"[S-JEPA] unavailable; sjepa/fusion branches will be skipped: {exc}")

def extract_sjepa_embeddings(X):
    if not SJEPA_AVAILABLE:
        raise RuntimeError("S-JEPA model is not available.")
    target = get_submodule_by_suffix(SJEPA_MODEL, SJEPA["embedding_module"])
    if target is None:
        raise RuntimeError(f"Could not find S-JEPA module named/suffixed {SJEPA['embedding_module']!r}")

    store = {}
    def hook(module, inputs):
        z = inputs[0]
        if isinstance(z, (tuple, list)):
            z = z[0]
        store["z"] = z.detach()

    h = target.register_forward_pre_hook(hook)
    outs = []
    try:
        with torch.no_grad():
            for i in range(0, len(X), int(SJEPA["embedding_batch"])):
                xb = torch.as_tensor(np.asarray(X[i:i+int(SJEPA["embedding_batch"])]), dtype=torch.float32, device=DEVICE)
                _ = SJEPA_MODEL(xb)
                z = store["z"]
                if z.ndim == 3:
                    if SJEPA["embedding_pool"] == "mean":
                        z = z.mean(dim=1)
                    elif SJEPA["embedding_pool"] == "flatten":
                        z = z.reshape(z.shape[0], -1)
                    else:
                        raise ValueError(f"Unsupported embedding_pool={SJEPA['embedding_pool']}")
                elif z.ndim > 3:
                    z = z.reshape(z.shape[0], -1)
                outs.append(z.float().cpu().numpy())
    finally:
        h.remove()

    return np.concatenate(outs, axis=0)

def compute_branch_features_from_X(X, branch):
    if branch == "riemann":
        return tangent_at_identity(trial_covariances(X))
    if branch == "sjepa":
        return extract_sjepa_embeddings(X)
    if branch == "fusion":
        tan = tangent_at_identity(trial_covariances(X))
        emb = extract_sjepa_embeddings(X)
        return np.concatenate([tan, emb], axis=1)
    raise ValueError(f"Unknown branch={branch}")

branches = [b for b in ADAPT["branches"] if b == "riemann" or SJEPA_AVAILABLE]
print("branches that will run:", branches)


S-JEPA official pretrained model loaded: braindecode/signal-jepa_without-chans
branches that will run: ['riemann', 'sjepa', 'fusion']


## 7. Precompute source-side features only


In [9]:
# Source-subject EA is allowed because source data are fully available training data.
# This cache is safe for target evaluation because the target subject is always excluded from the source pool.

X_SOURCE_EA = np.empty_like(X_ALL, dtype=np.float64)

for sid in subjects_all:
    rows = np.where(SUBJ == sid)[0]
    P = fit_ea_transform(X_ALL[rows])
    X_SOURCE_EA[rows] = apply_ea_transform(X_ALL[rows], P)

SOURCE_FEATURES = {}
for branch in branches:
    print(f"Computing source-cache features for branch={branch} ...")
    SOURCE_FEATURES[branch] = compute_branch_features_from_X(X_SOURCE_EA, branch)
    print(f"  {branch}: {SOURCE_FEATURES[branch].shape}")

print("Source feature cache ready.")


Computing source-cache features for branch=riemann ...
  riemann: (2000, 435)
Computing source-cache features for branch=sjepa ...
  sjepa: (2000, 64)
Computing source-cache features for branch=fusion ...
  fusion: (2000, 499)
Source feature cache ready.


## 8. Calibration/evaluation helpers


In [10]:
_TARGET_FEATURE_CACHE = {}

def _hash_transform(P):
    if P is None:
        return "identity"
    return hashlib.md5(np.asarray(P, dtype=np.float32).tobytes()).hexdigest()[:12]

def get_target_features(rows, branch, P=None):
    rows = np.asarray(rows, dtype=int)
    key = (branch, tuple(rows.tolist()), _hash_transform(P))
    if key in _TARGET_FEATURE_CACHE:
        return _TARGET_FEATURE_CACHE[key]

    X = X_ALL[rows]
    if P is not None:
        X = apply_ea_transform(X, P)

    F = compute_branch_features_from_X(X, branch)
    _TARGET_FEATURE_CACHE[key] = F
    return F

def make_balanced_calibration_split(target_rows, K, rng):
    target_rows = np.asarray(target_rows, dtype=int)

    if K <= 0:
        return np.array([], dtype=int), target_rows.copy()

    classes = np.unique(Y_ALL[target_rows])
    if len(classes) != 2:
        raise ValueError("Expected binary target rows.")

    per_class = int(K) // 2
    if per_class < 1:
        raise ValueError("K must be 0 or at least 2 for a binary balanced split.")

    cal_parts = []
    for c in classes:
        class_rows = target_rows[Y_ALL[target_rows] == c].copy()
        rng.shuffle(class_rows)
        take = min(per_class, len(class_rows))
        cal_parts.append(class_rows[:take])

    cal = np.concatenate(cal_parts).astype(int)
    cal_set = set(cal.tolist())
    test = np.asarray([r for r in target_rows if r not in cal_set], dtype=int)

    return cal, test

def sample_source_pool(pool_rows, source_cap, rng):
    pool_rows = np.asarray(pool_rows, dtype=int)

    if source_cap == "all":
        return pool_rows.copy()

    cap = int(source_cap)
    if cap >= len(pool_rows):
        return pool_rows.copy()

    classes = np.unique(Y_ALL[pool_rows])
    per_class = max(cap // len(classes), 1)
    parts = []

    for c in classes:
        class_rows = pool_rows[Y_ALL[pool_rows] == c].copy()
        rng.shuffle(class_rows)
        parts.append(class_rows[:min(per_class, len(class_rows))])

    chosen = np.concatenate(parts).astype(int)

    # Fill remainder, if cap is not divisible by number of classes.
    if len(chosen) < cap:
        used = set(chosen.tolist())
        rest = np.asarray([r for r in pool_rows if r not in used], dtype=int)
        rng.shuffle(rest)
        chosen = np.concatenate([chosen, rest[:cap - len(chosen)]])

    return chosen[:cap]

def fit_predict_blocks(X_train, y_train, X_test, sample_weight=None):
    if len(np.unique(y_train)) < 2:
        raise ValueError("Cannot train classifier with fewer than 2 classes.")

    if ADAPT["standardize"]:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    clf = LogisticRegression(
        C=float(ADAPT["logreg_C"]),
        max_iter=int(ADAPT["logreg_max_iter"]),
        solver="lbfgs",
    )
    clf.fit(X_train, y_train, sample_weight=sample_weight)
    return clf.predict(X_test)

def score_prediction(y_true, y_pred):
    return float(balanced_accuracy_score(y_true, y_pred))


## 9. Run corrected adaptive-calibration sweep


In [11]:
targets = subjects_all if ADAPT["target_subjects"] == "all" else [str(s) for s in ADAPT["target_subjects"] if str(s) in set(SUBJ)]
print(f"Targets ({len(targets)}): {targets}")

all_rows = np.arange(len(Y_ALL))
results = []

for target in targets:
    target_rows = np.where(SUBJ == str(target))[0]
    source_pool_all = np.where(SUBJ != str(target))[0]

    for K in ADAPT["calibration_grid"]:
        for rep in range(int(ADAPT["n_repeats"])):
            rng_split = np.random.default_rng(int(RUN["seed"]) + 7919 * int(target) + 101 * int(K) + rep)
            cal_rows, test_rows = make_balanced_calibration_split(target_rows, int(K), rng_split)

            if len(test_rows) == 0 or len(np.unique(Y_ALL[test_rows])) < 2:
                continue

            # Fold-safe target EA: fit on calibration only.
            # No target test trial is used. If K=0, target-EA is unavailable.
            P_target_cal = fit_ea_transform(X_ALL[cal_rows]) if len(cal_rows) else None

            for branch in branches:
                # 1) within_only: only target calibration trials.
                if len(cal_rows) > 0 and len(np.unique(Y_ALL[cal_rows])) == 2:
                    F_target = get_target_features(np.concatenate([cal_rows, test_rows]), branch, P=P_target_cal)
                    F_cal = F_target[:len(cal_rows)]
                    F_test = F_target[len(cal_rows):]

                    try:
                        pred = fit_predict_blocks(F_cal, Y_ALL[cal_rows], F_test)
                        results.append({
                            "target": str(target),
                            "branch": branch,
                            "strategy": "within_only",
                            "K": int(K),
                            "rep": int(rep),
                            "source_cap": "none",
                            "calibration_upweight": "none",
                            "n_source_train": 0,
                            "n_cal_train": int(len(cal_rows)),
                            "n_test": int(len(test_rows)),
                            "target_ea": "calibration_only",
                            "balanced_accuracy": score_prediction(Y_ALL[test_rows], pred),
                        })
                    except Exception as exc:
                        results.append({
                            "target": str(target), "branch": branch, "strategy": "within_only",
                            "K": int(K), "rep": int(rep), "source_cap": "none",
                            "calibration_upweight": "none", "n_source_train": 0,
                            "n_cal_train": int(len(cal_rows)), "n_test": int(len(test_rows)),
                            "target_ea": "calibration_only", "balanced_accuracy": np.nan,
                            "error": str(exc),
                        })

                # 2) transfer strategies: source cap sweep.
                for source_cap in ADAPT["source_cap_grid"]:
                    rng_src = np.random.default_rng(int(RUN["seed"]) + 33331 * int(target) + 499 * int(K) + 37 * rep + int(hashlib.md5(str(source_cap).encode()).hexdigest()[:6], 16))
                    source_rows = sample_source_pool(source_pool_all, source_cap, rng_src)

                    F_source = SOURCE_FEATURES[branch][source_rows]
                    y_source = Y_ALL[source_rows]

                    # transfer_only: strict zero-shot. It does not use target calibration
                    # to fit target EA. Target test features use identity/no target EA.
                    F_test_zero = get_target_features(test_rows, branch, P=None)
                    try:
                        pred = fit_predict_blocks(F_source, y_source, F_test_zero)
                        results.append({
                            "target": str(target),
                            "branch": branch,
                            "strategy": "transfer_only",
                            "K": int(K),
                            "rep": int(rep),
                            "source_cap": str(source_cap),
                            "calibration_upweight": "none",
                            "n_source_train": int(len(source_rows)),
                            "n_cal_train": 0,
                            "n_test": int(len(test_rows)),
                            "target_ea": "none",
                            "balanced_accuracy": score_prediction(Y_ALL[test_rows], pred),
                        })
                    except Exception as exc:
                        results.append({
                            "target": str(target), "branch": branch, "strategy": "transfer_only",
                            "K": int(K), "rep": int(rep), "source_cap": str(source_cap),
                            "calibration_upweight": "none", "n_source_train": int(len(source_rows)),
                            "n_cal_train": 0, "n_test": int(len(test_rows)), "target_ea": "none",
                            "balanced_accuracy": np.nan, "error": str(exc),
                        })

                    # transfer_plus_cal: source + labeled target calibration.
                    if len(cal_rows) > 0 and len(np.unique(Y_ALL[cal_rows])) == 2:
                        F_target = get_target_features(np.concatenate([cal_rows, test_rows]), branch, P=P_target_cal)
                        F_cal = F_target[:len(cal_rows)]
                        F_test = F_target[len(cal_rows):]

                        X_train_base = np.vstack([F_source, F_cal])
                        y_train_base = np.concatenate([y_source, Y_ALL[cal_rows]])

                        for upweight in ADAPT["calibration_upweight_grid"]:
                            sw = np.concatenate([
                                np.ones(len(source_rows), dtype=np.float64),
                                np.full(len(cal_rows), float(upweight), dtype=np.float64),
                            ])
                            try:
                                pred = fit_predict_blocks(X_train_base, y_train_base, F_test, sample_weight=sw)
                                results.append({
                                    "target": str(target),
                                    "branch": branch,
                                    "strategy": "transfer_plus_cal",
                                    "K": int(K),
                                    "rep": int(rep),
                                    "source_cap": str(source_cap),
                                    "calibration_upweight": str(upweight),
                                    "n_source_train": int(len(source_rows)),
                                    "n_cal_train": int(len(cal_rows)),
                                    "n_test": int(len(test_rows)),
                                    "target_ea": "calibration_only",
                                    "balanced_accuracy": score_prediction(Y_ALL[test_rows], pred),
                                })
                            except Exception as exc:
                                results.append({
                                    "target": str(target), "branch": branch, "strategy": "transfer_plus_cal",
                                    "K": int(K), "rep": int(rep), "source_cap": str(source_cap),
                                    "calibration_upweight": str(upweight),
                                    "n_source_train": int(len(source_rows)), "n_cal_train": int(len(cal_rows)),
                                    "n_test": int(len(test_rows)), "target_ea": "calibration_only",
                                    "balanced_accuracy": np.nan, "error": str(exc),
                                })

RES = pd.DataFrame(results)
RES.to_csv(ARTIFACT_DIR / "calibration_transfer_sweep_results.csv", index=False)

print("Finished sweep.")
print("Rows:", len(RES))
print("Errors:", int(RES["balanced_accuracy"].isna().sum()) if "balanced_accuracy" in RES else "n/a")
display(RES.head())


Targets (6): ['7', '22', '23', '28', '40', '44']
Finished sweep.
Rows: 33300
Errors: 0


,target,branch,strategy,K,rep,source_cap,calibration_upweight,n_source_train,n_cal_train,n_test,target_ea,balanced_accuracy
0,7,riemann,transfer_only,0,0,24,none,24,0,40,none,0.400
1,7,riemann,transfer_only,0,0,48,none,48,0,40,none,0.450
2,7,riemann,transfer_only,0,0,96,none,96,0,40,none,0.500
3,7,riemann,transfer_only,0,0,200,none,200,0,40,none,0.550
4,7,riemann,transfer_only,0,0,all,none,1960,0,40,none,0.475


## 10. Summaries and plots


In [ ]:
if RES.empty:
    raise RuntimeError("No results to summarize.")

# Mean for every actual setting.
mean_setting = (
    RES.dropna(subset=["balanced_accuracy"])
       .groupby(["branch", "strategy", "K", "source_cap", "calibration_upweight"], as_index=False)
       .agg(
           mean_balanced_accuracy=("balanced_accuracy", "mean"),
           std_balanced_accuracy=("balanced_accuracy", "std"),
           n=("balanced_accuracy", "size"),
       )
)
mean_setting.to_csv(ARTIFACT_DIR / "calibration_mean_by_setting.csv", index=False)

# Best transfer_plus_cal setting per branch/K across the sweep.
tpc = mean_setting[mean_setting["strategy"] == "transfer_plus_cal"].copy()
best_tpc = (
    tpc.sort_values(["branch", "K", "mean_balanced_accuracy"], ascending=[True, True, False])
       .groupby(["branch", "K"], as_index=False)
       .head(1)
       .reset_index(drop=True)
)
best_tpc.to_csv(ARTIFACT_DIR / "best_transfer_plus_cal_by_branch_K.csv", index=False)

# Best transfer_only source-cap per branch/K.
to = mean_setting[mean_setting["strategy"] == "transfer_only"].copy()
best_to = (
    to.sort_values(["branch", "K", "mean_balanced_accuracy"], ascending=[True, True, False])
      .groupby(["branch", "K"], as_index=False)
      .head(1)
      .reset_index(drop=True)
)
best_to.to_csv(ARTIFACT_DIR / "best_transfer_only_by_branch_K.csv", index=False)

# within_only mean curve.
within = (
    RES.dropna(subset=["balanced_accuracy"])
       .query("strategy == 'within_only'")
       .groupby(["branch", "K"], as_index=False)
       .agg(mean_balanced_accuracy=("balanced_accuracy", "mean"))
)
within["curve"] = "within_only"

best_tpc_curve = best_tpc[["branch", "K", "mean_balanced_accuracy", "source_cap", "calibration_upweight"]].copy()
best_tpc_curve["curve"] = "transfer_plus_cal_best_sweep"

best_to_curve = best_to[["branch", "K", "mean_balanced_accuracy", "source_cap", "calibration_upweight"]].copy()
best_to_curve["curve"] = "transfer_only_best_source_cap"

plot_curve = pd.concat([
    within.assign(source_cap="none", calibration_upweight="none"),
    best_to_curve,
    best_tpc_curve,
], ignore_index=True)
plot_curve.to_csv(ARTIFACT_DIR / "calibration_curve_best_sweep.csv", index=False)

Kmax = max(ADAPT["calibration_grid"])
headline = plot_curve[plot_curve["K"] == Kmax].pivot_table(
    index="branch", columns="curve", values="mean_balanced_accuracy", aggfunc="first"
)
print(f"================ headline at K={Kmax} ================")
print((100 * headline).round(1).to_string())

print("\nBest transfer_plus_cal settings at Kmax:")
display(best_tpc[best_tpc["K"] == Kmax].sort_values("branch"))

summary = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "targets": targets,
    "branches": branches,
    "K_max": int(Kmax),
    "headline_at_Kmax_percent": (100 * headline).round(2).to_dict(),
    "best_transfer_plus_cal_at_Kmax": best_tpc[best_tpc["K"] == Kmax].to_dict(orient="records"),
    "notes": [
        "target EA for within_only and transfer_plus_cal is fitted only on target calibration trials",
        "transfer_only is strict zero-shot and uses no target EA",
        "S-JEPA branch uses official pretrained braindecode/signal-jepa_without-chans weights; no local embedding path",
    ],
}
with open(ARTIFACT_DIR / "calibration_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Saved summary to {ARTIFACT_DIR / 'calibration_summary.json'}")


================ headline at K=24 ================
curve    transfer_only_best_source_cap  transfer_plus_cal_best_sweep  within_only
branch                                                                           
fusion                            51.4                          70.2         71.8
riemann                           50.2                          68.1         68.0
sjepa                             53.5                          57.4         52.3

Best transfer_plus_cal settings at Kmax:


,branch,strategy,K,source_cap,calibration_upweight,mean_balanced_accuracy,std_balanced_accuracy,n
4,fusion,transfer_plus_cal,24,24,100,0.702083,0.140164,60
9,riemann,transfer_plus_cal,24,24,10,0.681250,0.166636,60
14,sjepa,transfer_plus_cal,24,all,10,0.573958,0.127357,60


Saved summary to /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-subject-adaptive-calibration-slim/20260614_1409_10bb0fb8/calibration_summary.json


In [13]:
if HAVE_MPL:
    branches_to_plot = sorted(plot_curve["branch"].unique())
    fig, axes = plt.subplots(1, len(branches_to_plot), figsize=(5.2 * len(branches_to_plot), 4.2), squeeze=False)

    for j, branch in enumerate(branches_to_plot):
        ax = axes[0][j]
        sub = plot_curve[plot_curve["branch"] == branch].sort_values("K")
        for curve_name, s in sub.groupby("curve"):
            s_plot = s.copy()
            if curve_name == "within_only":
                s_plot = s_plot[s_plot["K"] > 0]
            ax.plot(s_plot["K"], s_plot["mean_balanced_accuracy"], marker="o", label=curve_name)

        ax.axhline(0.5, linestyle="--", linewidth=1)
        ax.axhline(0.7, linestyle=":", linewidth=1)
        ax.set_title(f"branch: {branch}")
        ax.set_xlabel("calibration trials K")
        ax.set_ylim(0.35, 0.85)
        if j == 0:
            ax.set_ylabel("balanced accuracy")
        ax.legend(fontsize=8)

    fig.suptitle("Corrected subject-adaptive calibration: best sweep curves")
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "calibration_curves_best_sweep.png", dpi=160)
    plt.close(fig)
    print(f"Saved plot to {ARTIFACT_DIR / 'calibration_curves_best_sweep.png'}")
else:
    print("matplotlib unavailable; plot skipped.")


Saved plot to /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-subject-adaptive-calibration-slim/20260614_1409_10bb0fb8/calibration_curves_best_sweep.png


## 11. How to read the corrected output

Use `calibration_curve_best_sweep.csv` and `calibration_curves_best_sweep.png` first.

What matters:

- If `transfer_plus_cal_best_sweep` beats `within_only`, transfer is actually helping after correction.
- If `transfer_plus_cal_best_sweep` still stays below or equal to `within_only`, then cross-subject source data is not reducing the calibration burden.
- If `fusion` beats `riemann`, the official S-JEPA representation is adding something useful.
- If `fusion` ≈ `riemann` and `sjepa` is weak, then the useful signal is still mostly Riemannian/covariance structure.
- `transfer_only` is now a stricter zero-shot test. It may look worse than before because it no longer uses test-subject trials to fit EA.

Important: the “best sweep” curve is exploratory because it selects the best upweight/source-cap setting using the evaluated results. For a final paper figure, pick one setting based on this diagnostic run, then rerun once with that fixed setting.
